# YOLOv8 DENTEX Grounding Tool Training (Cloud/Colab)

This notebook will clone our VLM-DENTAL repository, download the DENTEX dataset directly to the cloud via Hugging Face, convert it to YOLO format, and run a heavy training loop (e.g., 500 epochs on `yolov8m.pt`).

**Hardware Recommendation:** Switch your Colab Runtime to **T4 GPU** or **A100 GPU** before starting.

In [ ]:
!pip install ultralytics huggingface_hub python-dotenv pandas

In [ ]:
import os
from google.colab import userdata

# Add your Hugging Face Token in the 'Secrets' tab on the left sidebar in Colab (name it HF_TOKEN)
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Successfully loaded HF_TOKEN from Colab Secrets!")
except:
    print("Warning: HF_TOKEN not found in Colab Secrets. The dataset download will be very slow and unauthenticated.")

### 1. Clone the Repo & Prepare Data

In [ ]:
!git clone https://github.com/YOUR_GITHUB_USERNAME/VLM-DENTAL.git
%cd VLM-DENTAL

In [ ]:
# Download the 10GB dataset and extract it locally in the cloud
!python download_and_cleanup.py

In [ ]:
# Convert COCO annotations to YOLO format (Generates the 705 train images and 50 val images)
!python scripts/prepare_yolo_dataset.py

### 2. Train the Grounding Tool

In [ ]:
# We use yolov8m.pt (medium model) instead of nano, and run for 500 epochs to ensure high precision.
!yolo train data=data/yolo_dentex/dataset.yaml model=yolov8m.pt epochs=500 imgsz=640 batch=16 device=0 project=data/models name=grounding_tool_heavy

### 3. Download the Final Weights

In [ ]:
from google.colab import files

# Download the trained model back to your local machine
weight_path = "data/models/grounding_tool_heavy/weights/best.pt"
if os.path.exists(weight_path):
    files.download(weight_path)
else:
    print("Training did not finish successfully or weights not found.")